# NYC Urban Massing — LLM Request Generation and Parsing

Qwen3-8B prompt-based pipeline (no fine-tuning).

This notebook:
1. Loads real NYC block tables, samples, edges and quantiles.
2. Recreates the 10-block pilot selection.
3. Generates four request styles.
4. Parses requests back into structured constraints.
5. Evaluates and saves round-trip results.

**Pilot:** 10 NYC training blocks, 40 requests.

Numeric styles recover four attributes with fixed tolerances.
Qualitative and comparative styles recover FAR constraints,
not complete numeric tables.

The saved pilot results are in `../results/`.

The processed NYC dataset is not included in this repository.

In [ ]:
from pathlib import Path
import json, zipfile, shutil, time, re
import numpy as np
import pandas as pd
import torch

MODEL_NAME = "Qwen/Qwen3-8B"

# Change this path to your own local NYC data archive.
ZIP_PATH = Path(
    "/content/globalmapper_google_drive/"
    "MyDrive/GenAI/project/data.zip"
)

# Optional: use already-extracted files.
DATA_DIR = Path("/content/nyc_llm_data")

# Store new outputs outside the repository.
OUTPUT_DIR = Path("/content/nyc_llm_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED = [
    "blocks.parquet",
    "samples.parquet",
    "edges.parquet",
    "quantiles.json",
]

missing = [x for x in REQUIRED if not (DATA_DIR / x).exists()]

if missing:
    assert ZIP_PATH.exists(), (
        "Set ZIP_PATH to the shared NYC data.zip or "
        "provide extracted files in DATA_DIR."
    )

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH) as z:
        for filename in missing:
            member = "data/processed/city/" + filename
            with z.open(member) as src:
                with open(DATA_DIR / filename, "wb") as dst:
                    shutil.copyfileobj(src, dst)

blocks = pd.read_parquet(DATA_DIR / "blocks.parquet")
samples = pd.read_parquet(DATA_DIR / "samples.parquet")
edges = pd.read_parquet(DATA_DIR / "edges.parquet")

quantiles = json.loads(
    (DATA_DIR / "quantiles.json").read_text()
)

print("Blocks:", len(blocks))
print("Samples:", len(samples))
print("Edges:", len(edges))
print("CUDA:", torch.cuda.is_available())

In [ ]:
# Original pilot sample IDs, selected across height classes.
PILOT_IDS = [
    24567, 19374, 19983, 12704, 10410,
    11907, 15688, 7570, 1701, 1276
]

HEIGHT_CLASSES = {
    0: "1-2 floors",
    1: "3-5 floors",
    2: "6-12 floors",
    3: "13 or more floors"
}

def table_for_index(i):
    b = blocks.iloc[int(i)]
    return {
        "block_id": str(b["block_id"]),
        "coverage": round(float(b["coverage"]), 4),
        "far_fp": round(float(b["far_fp"]), 4),
        "hclass": int(b["hclass"]),
        "res_share": round(float(b["res_share"]), 4),
        "boro": str(b["boro"])
    }

adjacency = {}

for e in edges.itertuples(index=False):
    adjacency.setdefault(int(e.i), []).append(int(e.j))
    adjacency.setdefault(int(e.j), []).append(int(e.i))

pilot = []

for sample_id in PILOT_IDS:
    row = samples.loc[samples["sample_id"] == sample_id]
    assert len(row) == 1, f"Missing sample {sample_id}"
    row = row.iloc[0]

    assert row["split"] == "train"
    assert int(row["k"]) == 1
    assert not bool(row["ctx_other_split"])

    target_idx = int(row["targets"][0])
    target_row = blocks.iloc[target_idx]

    assert bool(target_row["valid"])
    assert int(target_row["hclass"]) in HEIGHT_CLASSES

    neighbors = [
        table_for_index(i)
        for i in adjacency.get(target_idx, [])
        if bool(blocks.iloc[i]["valid"])
        and str(blocks.iloc[i]["split"]) == "train"
    ]

    pilot.append({
        "sample_id": int(sample_id),
        "target": table_for_index(target_idx),
        "neighbors": neighbors
    })

Q33 = float(quantiles["far_fp"]["city"]["q33"])
Q66 = float(quantiles["far_fp"]["city"]["q66"])

def far_category(far):
    if far <= Q33:
        return "low"
    if far <= Q66:
        return "medium"
    return "high"

def choose_neighbor(record):
    target_far = record["target"]["far_fp"]
    eligible = [
        n for n in record["neighbors"]
        if n["far_fp"] > 0
        and abs(n["far_fp"] - target_far) > 0.05
    ]
    if not eligible:
        return None
    return max(
        eligible,
        key=lambda n: abs(n["far_fp"] - target_far)
    )

print("Pilot samples:", len(pilot))
print("Height classes:",
      [r["target"]["hclass"] for r in pilot])
print("FAR q33/q66:", Q33, Q66)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()

@torch.inference_mode()
def ask_qwen(system, user, max_new_tokens=220):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        [prompt], return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(
        output[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

def parse_json(text):
    text = text.strip()
    text = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "", text, flags=re.IGNORECASE
    ).strip()

    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise
        result = json.loads(match.group(0))

    if not isinstance(result, dict):
        raise ValueError("Expected JSON object")

    return result

print("Qwen3-8B ready")

In [ ]:
NUMERIC_GENERATOR = '''
You are an urban planner generating NYC massing requests.
Preserve all numerical attributes and block ID.
Express hclass using its corresponding floor range:
0=1-2, 1=3-5, 2=6-12, 3=13+ floors.
Do not invent additional constraints.
Output only one English request.
'''

NUMERIC_PARSER = '''
Parse the urban massing request into JSON with keys:
block_id, coverage, far_fp, hclass, res_share.
Coverage and residential share are fractions.
Height mapping: 1-2 floors=0, 3-5=1, 6-12=2,
13+ floors=3. Missing values must be null.
Output valid JSON only.
'''

QUAL_GEN = '''
Write one NYC urban massing request using only the given
block ID and low/medium/high FAR category.
Do not reveal an exact FAR or invent other constraints.
'''

QUAL_PARSE = f'''
Parse one request into JSON with keys block_id, far_category.
Allowed categories: low, medium, high, null.
City-wide FAR q33={Q33:.4f}, q66={Q66:.4f}.
Low means <=q33; medium means >q33 and <=q66;
high means >q66. Do not guess exact FAR.
Output JSON only.
'''

COMP_GEN = '''
Write one urban planning request comparing the target
block's FAR against the named neighboring block.
Preserve both IDs and the higher/lower relationship.
Do not reveal target FAR or invent other constraints.
'''

COMP_PARSE = '''
Parse one comparative request to JSON with keys:
block_id, reference_block_id, far_relation.
far_relation must be higher, lower, equal or null.
The relation is for target relative to reference.
Output JSON only.
'''

TOLS = {
    "coverage": 0.01,
    "far_fp": 0.05,
    "hclass": 0,
    "res_share": 0.01
}

def numeric_accepted(gt, parsed):
    if parsed.get("block_id") != gt["block_id"]:
        return False

    for key, tol in TOLS.items():
        try:
            if abs(float(parsed[key]) - float(gt[key])) > tol:
                return False
        except (KeyError, TypeError, ValueError):
            return False

    return True

def run_one(record, style):
    gt = record["target"]

    if style in ("exact", "natural"):
        payload = {
            "block_id": gt["block_id"],
            **{k: gt[k] for k in TOLS},
            "height_description": HEIGHT_CLASSES[gt["hclass"]]
        }

        instruction = (
            "Use direct explicit numeric constraints."
            if style == "exact" else
            "Use fluent natural urban planning language, "
            "while retaining all precise numeric constraints."
        )

        request = ask_qwen(
            NUMERIC_GENERATOR,
            instruction + "\n" + json.dumps(payload)
        )

        raw = ask_qwen(
            NUMERIC_PARSER,
            "Parse this request:\n" + request
        )

        expected = payload

    elif style == "qualitative":
        expected = {
            "block_id": gt["block_id"],
            "far_category": far_category(gt["far_fp"])
        }

        request = ask_qwen(
            QUAL_GEN, json.dumps(expected)
        )

        raw = ask_qwen(
            QUAL_PARSE,
            "Parse this request:\n" + request
        )

    elif style == "comparative":
        neighbor = choose_neighbor(record)
        if neighbor is None:
            return None

        relation = (
            "higher" if gt["far_fp"] > neighbor["far_fp"]
            else "lower"
        )

        expected = {
            "block_id": gt["block_id"],
            "reference_block_id": neighbor["block_id"],
            "far_relation": relation
        }

        request = ask_qwen(
            COMP_GEN, json.dumps(expected)
        )

        raw = ask_qwen(
            COMP_PARSE,
            "Parse this request:\n" + request
        )

    else:
        raise ValueError(style)

    try:
        parsed = parse_json(raw)
        valid_json = True

        if style in ("exact", "natural"):
            accepted = numeric_accepted(gt, parsed)
        else:
            accepted = all(
                parsed.get(k) == v
                for k, v in expected.items()
            )

        error = None

    except Exception as exc:
        parsed = None
        valid_json = False
        accepted = False
        error = str(exc)

    return {
        "sample_id": record["sample_id"],
        "block_id": gt["block_id"],
        "style": style,
        "ground_truth": expected,
        "request": request,
        "raw_parser_output": raw,
        "parsed": parsed,
        "json_valid": valid_json,
        "accepted": accepted,
        "error": error
    }

In [ ]:
STYLES = [
    "exact", "natural",
    "qualitative", "comparative"
]

results = []
start = time.time()

for style in STYLES:
    for record in pilot:
        result = run_one(record, style)

        if result is None:
            continue

        results.append(result)
        print(
            style,
            result["block_id"],
            "accepted=", result["accepted"]
        )

df = pd.DataFrame(results)

summary = df.groupby("style").agg(
    generated=("accepted", "size"),
    valid_json=("json_valid", "sum"),
    accepted=("accepted", "sum")
).reset_index()

summary["acceptance_rate"] = (
    summary["accepted"] / summary["generated"]
)

print("\n", summary.to_string(index=False))
print("Elapsed:", round(time.time() - start, 1), "s")

(OUTPUT_DIR / "roundtrip_results.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False)
)

summary.to_csv(
    OUTPUT_DIR / "evaluation_summary.csv",
    index=False
)

print("Saved to:", OUTPUT_DIR)